# Deep Scan: Primary vs V2 Columns

This notebook investigates the relationship between the primary environmental columns (e.g., `doc`, `salt`, `do_`) and their "V2" counterparts (e.g., `doc2`, `salt2`, `do_2`) natively stored in the raw shapefiles. 

Previous logic in `utils/data_loader.py` assumed the V2 columns were superior because they had higher `.notna()` valid counts and were pre-cast as floats. However, there were suspicions that the extra "valid" data points in the V2 columns were actually just missing data placeholders coded as `0.0` or Limit of Detection (LOD) thresholds.

A background deep scan script (`76_v1_v2_deep_scan.py`) was executed to check:
1. When both columns have valid (non-null, non-zero) data, are they identical?
2. When the primary column is missing (`NaN`), is the V2 column `0.0`?

This scan evaluated **all parameters across all basins, depth layers, and seasons**.


In [ ]:
import pandas as pd
import numpy as np

# Load the deep scan results
results_df = pd.read_csv('76_v1_v2_deep_scan_results.csv')
results_df.head()


### 1. Overall Aggregated Findings

Let's look at the overall matching rates across all basins and seasons. 
- **Match Rate**: The percentage of times V1 equals V2 when both contain valid measurements.
- **Zero-Fill Rate**: The percentage of times V2 is exactly `0.0` when V1 is legitimately missing (`NaN`/`None`).


In [ ]:
agg_df = results_df.groupby('parameter').agg({
    'v1_valid_count': 'sum',
    'v2_valid_count': 'sum',
    'both_valid_count': 'sum',
    'match_count': 'sum',
    'v1_missing_count': 'sum',
    'v2_is_zero_when_v1_missing': 'sum'
}).reset_index()

agg_df['Match Rate (%)'] = (agg_df['match_count'] / agg_df['both_valid_count'] * 100).round(2)
agg_df['Zero-Fill Rate (%)'] = (agg_df['v2_is_zero_when_v1_missing'] / agg_df['v1_missing_count'] * 100).round(2)

display(agg_df[['parameter', 'v1_valid_count', 'v2_valid_count', 'Match Rate (%)', 'Zero-Fill Rate (%)']].sort_values('Zero-Fill Rate (%)', ascending=False))


### 2. Breakdown by Parameter

**Dissolved Organic Carbon (DOC), Salinity (Salt), TOC, and pH:**
For these parameters, the zero-fill rate is **100%**. Every single time the primary column is missing, the V2 column replaces it with an artificial `0.0`. Additionally, the match rate when both are present is nearly 100% (or 92% for salt, likely due to a minor scaling conversion or rounding difference). This proves beyond doubt that `doc2` is literally just `doc` with missing data overridden.

**Nutrients (TN, TP, COD) and Biological (Chlorophyll):**
The zero-fill rates hover between 80% and 90%. While not 100%, an overwhelming majority of the "extra" data points in the V2 columns are artificial zeroes.

### 3. What about the remaining 10-20% in Nutrients? (Limit of Detection)
For parameters like Ammonia Nitrogen (`nh4n`) or Nitrate Nitrogen (`no3n`), the zero-fill rate dropped to ~56-65%. This meant the V2 column was providing a valid non-zero number while the primary column was blank.

Further investigation into these "true extra" values revealed they are almost entirely **Limit of Detection (LOD)** placeholders. For example, out of 8,294 "extra" measurements for `nh4n2`, 8,234 of them were exactly `0.01`. 

Because the V2 columns are forced into a float format, they cannot store string markers like `<0.01` (Below Detection Limit). Instead, the database administrators hardcoded the detection threshold (`0.01`) as an absolute value. Meanwhile, the primary columns correctly left these censored values as `NaN` or strings that convert to `NaN`.

### Conclusion
The hypothesis is definitively confirmed: **The V2 columns are not superior datasets. They are simply the primary columns with missing data masked as `0.0` or Limit of Detection placeholders (`0.01`).**

By rewriting the data loader to parse the primary string columns directly (`pd.to_numeric(..., errors='coerce')`), we safely ingest the data while naturally generating `NaN` values, bypassing the artificial zero and LOD corruption altogether.
